# Visual Caption Generation

Generates a detailed caption for every **semantic** (non-decorative)
visual in one PDF's dual-pipeline-merged output — photos, labeled
diagrams, charts, and tables all go through **Qwen2.5-VL**, since the
goal here is a detailed, well-grounded description rather than a
clinical/diagnostic reading.

Runs against `output/{PDF_STEM}/auto/{PDF_STEM}_dual_pipeline_merged.json`
(see `dual_pipeline_parsing.ipynb`), which already tags each visual
`relevance: "semantic" | "decorative"` — decorative visuals are
skipped entirely, both for captioning and in the final unified
document.

**No OCR-label grounding**: unlike the MinerU content_list, the merged
document doesn't carry MinerU's leader-line-label `content` field, so
captions here are generated from the image alone.

**Hardware note:** runs on Apple Silicon (MPS, no CUDA). Uses bfloat16 and
loads to CPU before `.to("mps")` — torch's MPS backend has unresolved
SIGSEGVs in its fp16 cast kernel and in device_map-based loading (see
`src/captioning/qwen_vl.py` docstring for issue links).

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from captioning.qwen_vl import QwenVLCaptioner

PROJECT_ROOT

/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

## Load dual-pipeline-merged output for one PDF

In [ ]:
PDF_STEM = "Anatomy of Neck - Basic of DEMN.pdf_origin"
OUTPUT_DIR = PROJECT_ROOT / "output" / PDF_STEM / "auto"
MERGED_PATH = OUTPUT_DIR / f"{PDF_STEM}_dual_pipeline_merged.json"

with open(MERGED_PATH) as f:
    merged_document = json.load(f)

# Flatten every semantic (non-decorative) visual into one list, in page order.
visual_items = [
    {"page_idx": int(page_idx), **visual}
    for page_idx, page in merged_document.items()
    for visual in page.get("visuals", [])
    if visual.get("relevance") == "semantic"
]

len(visual_items), visual_items[0] if visual_items else None

## Smoke test on one labeled diagram

In [ ]:
captioner = QwenVLCaptioner("Qwen/Qwen2.5-VL-7B-Instruct")

sample = visual_items[0]
image_path = OUTPUT_DIR / sample["img_path"]
caption = captioner.caption(str(image_path))

print(json.dumps({
    "image_path": str(image_path),
    "content_type": sample["type"],
    "sub_type": sample.get("sub_type"),
    "caption": caption,
}, indent=2))

## Batch run over all semantic visuals in the PDF

In [ ]:
from tqdm.auto import tqdm

results = []

for item in tqdm(visual_items, desc="Captioning visuals"):
    image_path = OUTPUT_DIR / item["img_path"]
    caption = captioner.caption(str(image_path))
    results.append({
        "item_id": item["item_id"],
        "image_path": str(image_path),
        "content_type": item["type"],
        "sub_type": item.get("sub_type"),
        "caption": caption,
    })

captioner.unload()
len(results)

In [ ]:
results_path = OUTPUT_DIR / f"{PDF_STEM}_stage1_captions.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

results_path

## Unify captions with document text

Merges Stage 1 captions back into the merged document, replacing each
semantic visual with a `[FIGURE:item_id]` marker + caption, inlined in
page order. Decorative visuals were dropped before captioning, so they
don't appear here either. `item_id` is what ties an image to its
caption — see `src/ingestion/unify.py::build_unified_items_from_merged`.

In [ ]:
from ingestion.unify import build_unified_items_from_merged, render_unified_text, render_unified_markdown

captions = {r["item_id"]: r["caption"] for r in results}
unified_items = build_unified_items_from_merged(merged_document, captions)
unified_text = render_unified_text(unified_items)

print(f"{len(unified_items)} unified items, {len(unified_text)} chars")
print(unified_text[:1500])

In [ ]:
unified_text_path = OUTPUT_DIR / f"{PDF_STEM}_unified_text.txt"
with open(unified_text_path, "w") as f:
    f.write(unified_text)

unified_text_path

In [ ]:
unified_markdown = render_unified_markdown(unified_items)

unified_md_path = OUTPUT_DIR / f"{PDF_STEM}_unified_text.md"
with open(unified_md_path, "w") as f:
    f.write(unified_markdown)

unified_md_path